In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

caminho_arquivo = r"/home/jc/Projetos/estudos-python/Postegre/origem_dados/V_OCORRENCIA_AMPLA.json"
df = pd.read_json(caminho_arquivo, encoding='utf-8-sig')

In [2]:
colunas = ["Numero_da_Ocorrencia", "Classificacao_da_Ocorrência", "Data_da_Ocorrencia","Municipio","UF","Regiao","Nome_do_Fabricante"]
df = df[colunas]
df.rename(columns={'Classificacao_da_Ocorrência' : 'Classificacao_da_Ocorrencia'} ,inplace=True)
df.head(3)

,Numero_da_Ocorrencia,Classificacao_da_Ocorrencia,Data_da_Ocorrencia,Municipio,UF,Regiao,Nome_do_Fabricante
0,7762,Incidente,2018-03-21,SÃO PAULO,SP,Sudeste,AGUSTA
1,7759,Acidente,2018-03-14,MONTES CLAROS,MG,Sudeste,CESSNA AIRCRAFT
2,7758,Acidente,2018-01-26,INACIOLÂNDIA,GO,Centro-Oeste,CESSNA AIRCRAFT


In [3]:
df.dtypes

Numero_da_Ocorrencia           int64
Classificacao_da_Ocorrencia      str
Data_da_Ocorrencia               str
Municipio                        str
UF                               str
Regiao                           str
Nome_do_Fabricante               str
dtype: object

In [4]:
df['Data_da_Ocorrencia'] = pd.to_datetime(df['Data_da_Ocorrencia'])
df.dtypes

Numero_da_Ocorrencia                    int64
Classificacao_da_Ocorrencia               str
Data_da_Ocorrencia             datetime64[us]
Municipio                                 str
UF                                        str
Regiao                                    str
Nome_do_Fabricante                        str
dtype: object

In [5]:
from datetime import datetime

ano_aula = datetime.now().year - 2
print(ano_aula)

2024


In [6]:
df = df[df['Data_da_Ocorrencia'].dt.year == ano_aula] #usando ano fixo para aprendizado
df.head(10)

,Numero_da_Ocorrencia,Classificacao_da_Ocorrencia,Data_da_Ocorrencia,Municipio,UF,Regiao,Nome_do_Fabricante
5182,41518,Incidente,2024-01-02,RIO DE JANEIRO,RJ,Sudeste,BOEING COMPANY
5183,41571,Incidente,2024-01-02,SANTA LUZIA,MA,Nordeste,ROBINSON HELICOPTER
5251,41646,Incidente,2024-01-04,SALVADOR,BA,Nordeste,BOEING COMPANY
12458,41586,Incidente,2024-01-03,SÃO LUÍS,MA,Nordeste,AIRBUS S.A.S.
12604,41590,Incidente,2024-01-01,CHAPECÓ,SC,Sul,BOEING COMPANY


In [ ]:
from sqlalchemy import create_engine, Integer, String, Date, VARCHAR, text

dbname = 'python'
user = 'postgres'
password = '56789'
host = 'localhost'
port = '5432' 

conexao_str = f'postgresql://{user}:{password}@{host}:{port}/{dbname}'
engine = create_engine(conexao_str)

nome_tabela = 'anac_sqlalchemy'

# Deletando registros com base no ano atual
cursor = engine.connect()
delete = text(f'DELETE FROM {nome_tabela} WHERE EXTRACT(YEAR FROM "Data_da_Ocorrencia") = {ano_aula}')
cursor.execute(delete)
cursor.commit()

# Enviando o dataframe para o banco de dados
df.to_sql(nome_tabela, engine, index=False, if_exists='append',
dtype={
	'Numero_da_Ocorrencia' : Integer,
	'Classificacao_da_Ocorrencia' : VARCHAR(30),
	'Data_da_Ocorrencia' : Date
})

engine.dispose()
cursor.close()